# Bear + Discord on Google Colab
This notebook builds the headless Rust runner from this fork and starts the Discord control bot. Select a GPU runtime first. Add `DISCORD_BOT_TOKEN` in Colab Secrets.

In [ ]:
# System packages and Rust toolchain
!apt-get update -qq
!apt-get install -y -qq build-essential curl wget file libssl-dev libgtk-3-dev libayatana-appindicator3-dev librsvg2-dev patchelf libwebkit2gtk-4.1-dev
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] = '/root/.cargo/bin:' + os.environ['PATH']


In [ ]:
# Clone the user's fork and the integration branch
%cd /content
!git clone --branch feature/colab-discord-headless https://github.com/Nader-Geek/Bear-BOT.git
%cd /content/Bear-BOT


In [ ]:
# Download Bear's pinned local ONNX models and runtime (several hundred MB)
!bash scripts/fetch-models.sh
!bash scripts/fetch-runtime.sh
import glob, os
# fetch-runtime.sh deliberately assembles all runtime libraries directly in runtimes/lib.
ort_candidates = glob.glob('/content/Bear-BOT/runtimes/lib/libonnxruntime.so*')
if not ort_candidates:
    raise RuntimeError('ONNX Runtime was not installed. Check the fetch-runtime.sh output above; the expected library path is runtimes/lib/libonnxruntime.so.')
os.environ['BEAR_MODELS_DIR'] = '/content/Bear-BOT/models'
os.environ['BEAR_ORT_PATH'] = ort_candidates[0]
print('ONNX Runtime:', os.environ['BEAR_ORT_PATH'])


In [ ]:
# Build the headless runner. This uses Bear's existing Pipeline, not a Python rewrite.
!cargo build --release -p bear-runner


In [ ]:
# Install the Discord control plane
!pip -q install -r bot/requirements.txt
from google.colab import userdata
token = userdata.get('DISCORD_BOT_TOKEN')
assert token, 'Add DISCORD_BOT_TOKEN in Colab > Secrets and grant this notebook access.'
os.environ['DISCORD_BOT_TOKEN'] = token
os.environ['BEAR_RUNNER'] = '/content/Bear-BOT/target/release/bear-runner'


## Optional: local Qwen-Image-Edit-2511
Bear's existing sidecar supports Qwen-Image-Edit-2511 through SDNQ, but its quantized weights are roughly 18.16 GB before runtime memory. **The current headless runner does not yet pass the desktop app's validated sidecar settings**, so installing the weights here will not make `!clean` use Qwen. Do not download them until headless sidecar activation is implemented and the available Colab GPU has been checked.

In [ ]:
# Optional Qwen setup. This downloads ~18.16 GB of weights plus a 2 GB safety margin.
import json, shutil, sys, subprocess
assert sys.version_info >= (3, 11) and sys.version_info < (3, 13), 'Sidecar needs Python 3.11 or 3.12.'
!nvidia-smi
assert shutil.disk_usage('/content').free > 21_000_000_000, 'Not enough free disk for Qwen weights and cache.'
checkpoint = {
  'repo': 'Disty0/Qwen-Image-Edit-2511-SDNQ-uint4-svd-r32',
  'revision': 'a285fceef1439d72d60533e83cb6d8921748a666',
  'dir': 'qwen-image-edit-2511-sdnq',
  'bytes': 18162784045,
  'expected': 'model_index.json',
  'adapters': [
    {'repo': 'lightx2v/Qwen-Image-Edit-2511-Lightning', 'revision': 'd74eba145674fd7e31b949324e148e21e7118abd', 'file': 'Qwen-Image-Edit-2511-Lightning-8steps-V1.0-bf16.safetensors'},
    {'repo': 'tori29umai/QwenImageEdit2511_LoRA', 'revision': '3ca05a1ece5606d4de68ebc564d46713fd77e549', 'file': 'QIE2511_FukidashiErase_V1_dim4_1e-3-000080_dim1.safetensors'}
  ]
}
root = '/content/Bear-BOT/.sidecar-venv'
subprocess.check_call([sys.executable, 'sidecar/bootstrap.py', '--root', root, '--source', 'sidecar', '--backend', 'sdnq', '--accelerator', 'auto', '--model', 'qwen-image-edit-2511', '--checkpoint', json.dumps(checkpoint)])
os.environ['MANGA_CLEANER_SIDECAR'] = root
os.environ['MC_SIDECAR_WEIGHTS_ROOT'] = root + '/weights'
print('Qwen sidecar installed. Use !clean qwen or !clean-qwen with an attached image/ZIP.')


In [ ]:
# Start the bot. Keep this cell running while you use Discord.
!python bot/bear_bot.py
